In [2]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2://karenajvazan@localhost:5432/retailrocket")
journey = pd.read_sql("SELECT * FROM visitor_journey", engine)

print(journey.shape)
print(journey.dtypes)

(1407419, 8)
visitorid                            int64
first_view             datetime64[us, UTC]
first_cart             datetime64[us, UTC]
first_purchase         datetime64[us, UTC]
views_before_action                  int64
items_before_action                  int64
has_action                            bool
has_purchase                          bool
dtype: object


In [3]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

j = journey[journey["items_before_action"] >= 1].copy()
j["deep"] = j["items_before_action"] >= 2

g = j.groupby("deep")["has_action"].agg(["sum", "count"]).astype(int)
print(g)

success = np.array([g.loc[True, "sum"],   g.loc[False, "sum"]])
nobs    = np.array([g.loc[True, "count"], g.loc[False, "count"]])

rates = success / nobs
z, p = proportions_ztest(success, nobs)
ci_low, ci_high = confint_proportions_2indep(
    success[0], nobs[0], success[1], nobs[1], compare="diff"
)

print(f"2+ товара: {rates[0]:.2%},  1 товар: {rates[1]:.2%}")
print(f"Разница: {rates[0] - rates[1]:.2%} (95% ДИ: {ci_low:.2%} .. {ci_high:.2%})")
print(f"z = {z:.1f}, p = {p:.3g}")

         sum    count
deep                 
False  20682  1123416
True   12587   278436
2+ товара: 4.52%,  1 товар: 1.84%
Разница: 2.68% (95% ДИ: 2.60% .. 2.76%)
z = 83.2, p = 0


In [4]:
j24 = pd.read_sql("SELECT * FROM visitor_journey_24h WHERE items_24h >= 1", engine)
j24["deep"] = j24["items_24h"] >= 2

g24 = j24.groupby("deep")["action_24h"].agg(["sum", "count"]).astype(int)
print(g24)

s = np.array([g24.loc[True, "sum"],   g24.loc[False, "sum"]])
n = np.array([g24.loc[True, "count"], g24.loc[False, "count"]])
r = s / n
lo, hi = confint_proportions_2indep(s[0], n[0], s[1], n[1], compare="diff")

print(f"2+ товара: {r[0]:.2%},  1 товар: {r[1]:.2%}")
print(f"Разница: {r[0]-r[1]:.2%} (95% ДИ: {lo:.2%} .. {hi:.2%})")

         sum    count
deep                 
False  19834  1192874
True    8763   208978
2+ товара: 4.19%,  1 товар: 1.66%
Разница: 2.53% (95% ДИ: 2.44% .. 2.62%)


In [5]:
vd = pd.read_sql("SELECT is_weekend, bought FROM visitor_day", engine)

g = vd.groupby("is_weekend")["bought"].agg(["sum", "count"]).astype(int)
print(g)

s = np.array([g.loc[False, "sum"],   g.loc[True, "sum"]])
n = np.array([g.loc[False, "count"], g.loc[True, "count"]])
r = s / n
lo, hi = confint_proportions_2indep(s[0], n[0], s[1], n[1], compare="diff")

print(f"Будни: {r[0]:.3%},  выходные: {r[1]:.3%}")
print(f"Разница: {r[0]-r[1]:.3%} (95% ДИ: {lo:.3%} .. {hi:.3%})")

              sum    count
is_weekend                
False       10746  1240654
True         2401   385143
Будни: 0.866%,  выходные: 0.623%
Разница: 0.243% (95% ДИ: 0.213% .. 0.272%)
